# 🤖 LBot Translator V5.1 - Relational-Aware Self-Attention (RASA)

Este notebook treina um modelo GPT com **Relational-Aware Self-Attention** para traduzir comandos em português para **LBot Movement Language (LBML)**.

## 🎯 Novidade da V5.1 vs V5

| Aspecto | V5 | V5.1 | Melhoria |
|---------|----|----|----------|
| **Arquitetura** | GPT padrão | GPT + RASA | ✨ Atenção relacional |
| **Atenção** | Causal self-attention | Causal + Relational | ✨ Captura relações espaciais |
| **Relações** | Nenhuma | F↔frente, B↔trás, L↔esquerda, R↔direita | ✨ Consciência semântica |
| **Parâmetros** | ~1.5M | ~1.7M | ⚡ +15% para relações |
| **Dataset** | 40k exemplos | 40k exemplos | ✅ Idêntico (para comparação) |
| **Tempo de Treino** | ~15-20 min | ~18-25 min | ⚡ +20% por camada extra |

## 🧠 O que é RASA (Relational-Aware Self-Attention)?

**RASA** adiciona uma camada de atenção que aprende relações explícitas entre:
- **Tokens de entrada** (português): "frente", "trás", "esquerda", "direita", "gire", "ande"
- **Tokens de saída** (LBML): `F`, `B`, `L`, `R`, `D`, `R` (comandos)

A camada usa **bias relacional** para ensinar ao modelo que:
- "frente" → `F` são semanticamente relacionados
- "trás" → `B` são semanticamente relacionados  
- "esquerda" → `L` são semanticamente relacionados
- "direita" → `R` são semanticamente relacionados
- "gire" → `R` (rotação) são semanticamente relacionados

Isso melhora a tradução de **comandos compostos** onde a ordem e direção importam:
- ❌ V5: "vá frente e vire direita" pode confundir direções
- ✅ V5.1: Entende que "frente" e "vire direita" são ações sequenciais com direções distintas

## 📋 Formato LBML V4 (mantido)

**Deslocamento (Displacement):**
- Formato: `D<valor><direção>;`
- Direções: `F` (frente), `B` (trás), `L` (esquerda), `R` (direita)
- Exemplo: `D40F;` = Desloque 40cm para frente

**Rotação (Rotation):**
- Formato: `R<valor><direção>;`
- Direções: `L` (girar esquerda), `R` (girar direita)
- Exemplo: `R90R;` = Gire 90° para direita

**Comandos Compostos (máximo 3 ações):**
- Exemplo: `D40F;R90R;D20L;`
- Tradução: "Ande 40cm frente, gire 90° direita, ande 20cm esquerda"

## 🎯 Objetivo da V5.1

✅ **Comparar** se RASA melhora acurácia vs V5 base  
✅ **Manter** dataset idêntico (controle experimental)  
✅ **Adicionar** apenas camada relacional (mudança mínima)  
✅ **Avaliar** ganho em comandos compostos com múltiplas direções

## 📦 1. Instalação e Imports

In [ ]:
# Instalar dependências
!pip install torch numpy transformers datasets tiktoken wandb tqdm

import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import numpy as np
import os
import time
import re
from dataclasses import dataclass
from google.colab import files

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 📁 2. Carregamento e Processamento do Dataset V5

In [ ]:
# Upload do dataset V5
print("Faça upload do arquivo lbot_dataset_v5.txt:")
uploaded = files.upload()

# Carregar e processar dataset
with open('lbot_dataset_v5.txt', 'r', encoding='utf-8') as f:
    raw_data = f.read()

def parse_dataset(raw_data):
    """Extrai pares entrada-saída do dataset"""
    examples = []
    lines = raw_data.strip().split('\n')
    
    i = 0
    while i < len(lines):
        if lines[i].startswith('Entrada:'):
            entrada = lines[i].replace('Entrada:', '').strip()
            if i + 1 < len(lines) and lines[i + 1].startswith('Saída:'):
                saida = lines[i + 1].replace('Saída:', '').strip()
                examples.append((entrada, saida))
                i += 2
            else:
                i += 1
        else:
            i += 1
    return examples

# Processar dados
examples = parse_dataset(raw_data)
print(f"✅ Dataset carregado: {len(raw_data):,} caracteres")
print(f"✅ Exemplos extraídos: {len(examples):,}")

# Mostrar exemplos
print("\n📋 Primeiros 5 exemplos:")
for i in range(5):
    print(f"  {i+1}. '{examples[i][0]}' → '{examples[i][1]}'")

In [ ]:
# Criar dataset de treinamento formatado
def create_training_data(examples):
    """Formata dados para treinamento: 'comando -> codigo_lbml'"""
    training_text = ""
    for entrada, saida in examples:
        training_text += f"{entrada} -> {saida}\n"
    return training_text

# Criar e dividir dados (90% treino, 10% validação)
train_data = create_training_data(examples)
n = len(train_data)
train_data_final = train_data[:int(n*0.9)]
val_data = train_data[int(n*0.9):]

print(f"📊 Dados de treino: {len(train_data_final):,} caracteres")
print(f"📊 Dados de validação: {len(val_data):,} caracteres")

# Criar vocabulário
chars = sorted(list(set(train_data)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Funções de encode/decode
def encode_text(s):
    """Converte string para lista de índices"""
    return [stoi[c] for c in s]

def decode_text(l):
    """Converte lista de índices para string"""
    return ''.join([itos[i] for i in l])

# Criar referências globais
encode = encode_text
decode = decode_text

print(f"🔤 Vocabulário: {vocab_size} caracteres únicos")
print(f"🔤 Caracteres: {''.join(chars)}")

# Testar encode/decode
test_text = "vá 10 centímetros para frente -> D10F;"
encoded = encode(test_text)
decoded = decode(encoded)
print(f"\n🧪 Teste encode/decode:")
print(f"   Original: {test_text}")
print(f"   Encoded: {encoded[:10]}...")
print(f"   Decoded: {decoded}")
print(f"   ✅ {'Correto' if decoded == test_text else 'Erro'}")

## 🧠 3. Definição do Modelo GPT Otimizado

### 📊 Configuração V5.1 (Base V5 + RASA)

| Parâmetro | V5 | V5.1 | Motivo |
|-----------|----|----|--------|
| **block_size** | 128 | **128** = | Mantido |
| **vocab_size** | 70 | **70** = | Mantido |
| **n_layer** | 6 | **6** = | Mantido |
| **n_head** | 6 | **6** = | Mantido |
| **n_embd** | 384 | **384** = | Mantido |
| **dropout** | 0.2 | **0.2** = | Mantido |
| **learning_rate** | 1e-3 | **1e-3** = | Mantido |
| **max_iters** | 5,000 | **5,000** = | Mantido |
| **RASA** | ❌ | **✅ NOVA** | Camada relacional após cada atenção |

**Mudança arquitetural:**
- Cada bloco transformer agora tem: `LayerNorm → CausalSelfAttention → RASA → LayerNorm → MLP`
- RASA adiciona ~200K parâmetros (matriz de relações espaciais)

**Total de parâmetros:**
- V5: ~1.5M parâmetros
- V5.1: **~1.7M parâmetros** (+15% para relações espaciais)

In [ ]:
# Configuração do modelo V5.1 com RASA
@dataclass
class GPTConfig:
    block_size: int = 128      # Mantido do V5
    vocab_size: int = 70       # Mantido do V5
    n_layer: int = 6           # Mantido do V5
    n_head: int = 6            # Mantido do V5
    n_embd: int = 384          # Mantido do V5
    dropout: float = 0.2       # Mantido do V5
    bias: bool = True

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                           .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class RelationalSelfAttention(nn.Module):
    """
    RASA: Relational-Aware Self-Attention
    
    Adiciona consciência relacional para mapear tokens de entrada (português)
    aos tokens de saída (LBML), capturando relações espaciais:
    - "frente" ↔ F (forward)
    - "trás" ↔ B (backward) 
    - "esquerda" ↔ L (left)
    - "direita" ↔ R (right)
    - "gire" ↔ R (rotation)
    """
    def __init__(self, config):
        super().__init__()
        self.n_embd = config.n_embd
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        
        # Projeções para relational attention
        self.q_rel = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.k_rel = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.v_rel = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        
        # Matriz de bias relacional (aprendida durante treinamento)
        # Captura relações entre pares de tokens no vocabulário
        self.relation_bias = nn.Parameter(torch.zeros(config.vocab_size, config.vocab_size))
        
        # Projeção de saída
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
        
        # Máscara causal (mesma do CausalSelfAttention)
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                           .view(1, 1, config.block_size, config.block_size))
    
    def forward(self, x, input_ids=None):
        B, T, C = x.size()
        
        # Projetar Q, K, V para espaço relacional
        q = self.q_rel(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_rel(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.v_rel(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        # Atenção padrão: Q @ K^T / sqrt(d_k)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        
        # Adicionar bias relacional se input_ids fornecido
        # Isso cria conexões mais fortes entre tokens relacionados
        # Ex: "frente" (input) terá maior atenção para "F" (output)
        if input_ids is not None:
            # Extrair bias relacional para pares de tokens no batch
            rel_bias = self.relation_bias[input_ids.unsqueeze(-1), input_ids.unsqueeze(-2)]
            # rel_bias shape: (B, T, T)
            # Expandir para heads: (B, 1, T, T) e adicionar ao att
            att = att + rel_bias.unsqueeze(1)
        
        # Aplicar máscara causal
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)
        
        # Aplicar atenção aos valores
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.dropout(self.c_proj(y))
        
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    """
    Bloco Transformer V5.1 com RASA
    
    Arquitetura: 
    x → LayerNorm → CausalSelfAttention → residual → 
    x → LayerNorm → RelationalSelfAttention (RASA) → residual →
    x → LayerNorm → MLP → residual
    """
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        
        # NOVA: Camada de atenção relacional
        self.ln_rel = nn.LayerNorm(config.n_embd)
        self.rasa = RelationalSelfAttention(config)
        
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x, input_ids=None):
        # Atenção causal padrão
        x = x + self.attn(self.ln_1(x))
        
        # NOVA: Atenção relacional (RASA)
        x = x + self.rasa(self.ln_rel(x), input_ids)
        
        # MLP padrão
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        
        # MODIFICADO: Passar input_ids para RASA poder usar bias relacional
        for block in self.transformer.h:
            x = block(x, input_ids=idx)
        
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("✅ Modelo GPT V5.1 com RASA definido!")
print(f"📊 Configuração:")
print(f"   • Contexto: 128 tokens")
print(f"   • Vocabulário: 70 caracteres")
print(f"   • Camadas: 6 blocos transformer")
print(f"   • Cada bloco: CausalSelfAttention + RASA + MLP")
print(f"   • Dimensão: 384")
print(f"   • Cabeças: 6")
print(f"   • Dropout: 0.2")
print(f"   • ✨ NOVO: Relational-Aware Self-Attention")
print(f"   • ✨ Captura relações: frente↔F, trás↔B, esquerda↔L, direita↔R")

## 🏋️ 4. Treinamento do Modelo V5.1 com RASA

In [ ]:
# Preparar dados para treinamento
train_ids = np.array(encode(train_data_final), dtype=np.uint16)
val_ids = np.array(encode(val_data), dtype=np.uint16)

def get_batch(split, batch_size=32, block_size=128):
    """Cria batch de dados para treinamento"""
    data = train_ids if split == 'train' else val_ids
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if torch.cuda.is_available():
        x, y = x.cuda(), y.cuda()
    return x, y

# Configurar modelo V5.1
config = GPTConfig()
config.vocab_size = vocab_size
model = GPT(config)

if torch.cuda.is_available():
    model = model.cuda()
    print("🚀 Modelo movido para GPU")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Contar parâmetros da RASA especificamente
rasa_params = sum(p.numel() for name, p in model.named_parameters() if 'rasa' in name or 'relation_bias' in name)

print(f"📊 Parâmetros do modelo V5.1:")
print(f"   • Total: {total_params:,}")
print(f"   • Treináveis: {trainable_params:,}")
print(f"   • RASA (relacional): {rasa_params:,}")
print(f"   • Base (sem RASA): {total_params - rasa_params:,}")
print(f"📊 Tamanho estimado: {total_params * 4 / 1024 / 1024:.1f} MB")
print(f"💡 Aumento vs V5: {(total_params / 1_500_000 - 1) * 100:.1f}% mais parâmetros")

# Configurar otimizador
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

@torch.no_grad()
def estimate_loss():
    """Estima loss nos dados de treino e validação"""
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        losses_list = []
        for k in range(10):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses_list.append(loss.item())
        losses[split] = sum(losses_list) / len(losses_list)
    model.train()
    return losses

print("✅ Setup de treinamento V5.1 pronto!")
print("💡 Learning rate: 1e-3 (mesmo do V5 para comparação justa)")
print("💡 RASA aprenderá relações espaciais durante treinamento")

In [ ]:
# Treinamento principal - V5.1 com RASA
print("🚀 Iniciando treinamento V5.1 com Relational-Aware Self-Attention...\n")

model.train()
max_iters = 5000           # Mantido do V5 para comparação
eval_interval = 200        # Avalia a cada 200 iterações
log_interval = 100         # Log a cada 100

start_time = time.time()
best_val_loss = float('inf')

for iter in range(max_iters):
    # Avaliação periódica
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        elapsed = time.time() - start_time
        print(f"📊 Step {iter:4d} | Train: {losses['train']:.4f} | Val: {losses['val']:.4f} | Time: {elapsed:.1f}s")
        
        # Salvar melhor modelo
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            print(f"   ⭐ Novo melhor val_loss! Salvando checkpoint...")

    # Forward e backward pass
    X, Y = get_batch('train')
    logits, loss = model(X, Y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # Log do progresso
    if iter % log_interval == 0 and iter > 0:
        print(f"⚡ Iter {iter:4d} | Loss: {loss.item():.4f}")

print(f"\n✅ Treinamento V5.1 concluído em {time.time() - start_time:.1f}s!")
print(f"🎯 Melhor val_loss: {best_val_loss:.4f}")

# Salvar modelo V5.1
torch.save({
    'model': model.state_dict(),
    'config': config,
    'vocab_size': vocab_size,
    'stoi': stoi,
    'itos': itos,
    'train_loss': losses['train'],
    'val_loss': losses['val'],
    'best_val_loss': best_val_loss,
    'version': 'v5.1-RASA'
}, 'lbot_translator_v5-1.pt')

print("💾 Modelo salvo como 'lbot_translator_v5-1.pt'")
print(f"📊 Tamanho do modelo: {os.path.getsize('lbot_translator_v5-1.pt') / 1024 / 1024:.1f} MB")
print("\n🔬 Para comparar com V5:")
print("   • Use mesmo dataset e hiperparâmetros")
print("   • Compare val_loss e acurácia em comandos compostos")
print("   • RASA deve melhorar tradução de comandos com múltiplas direções")

## 🔄 5. Função para Carregar Modelo

In [ ]:
def load_lbot_model(path='lbot_translator_v5-1.pt'):
    """Carrega o modelo V5.1 com RASA treinado e recria as funções necessárias"""
    checkpoint = torch.load(path, map_location='cpu')
    
    # Recriar funções encode/decode
    stoi = checkpoint['stoi']
    itos = checkpoint['itos']
    
    def encode_text(s):
        return [stoi[c] for c in s]
    
    def decode_text(l):
        return ''.join([itos[i] for i in l])
    
    # Recriar modelo com RASA
    config = checkpoint['config']
    model = GPT(config)
    model.load_state_dict(checkpoint['model'])
    
    if torch.cuda.is_available():
        model = model.cuda()
    
    version = checkpoint.get('version', 'v5.1')
    print(f"✅ Modelo {version} carregado!")
    
    return model, encode_text, decode_text, stoi, itos

# Exemplo de uso:
# model, encode, decode, stoi, itos = load_lbot_model()

print("✅ Função de carregamento V5.1 definida!")
print("💡 Use: model, encode, decode, stoi, itos = load_lbot_model()")
print("💡 Compatível com modelo V5.1 (RASA incluída)")

## 🧪 6. Função de Tradução e Testes

In [ ]:
def lbot_translator(command, temperature=0.05, max_tokens=80):
    """
    Traduz comando em português para linguagem LBML V4 usando modelo V5.1 com RASA
    
    Args:
        command (str): Comando em português
        temperature (float): Controla aleatoriedade (menor = mais determinístico)
        max_tokens (int): Máximo de tokens a gerar
    
    Returns:
        str: Comando no formato LBML (ex: "D40F;R90R;D20L;")
    """
    model.eval()
    
    # Preparar input
    input_text = f"{command.strip()} ->"
    input_ids = torch.tensor(encode(input_text), dtype=torch.long).unsqueeze(0)
    
    if torch.cuda.is_available():
        input_ids = input_ids.cuda()
    
    # Gerar com temperatura baixa para mais precisão
    with torch.no_grad():
        generated = model.generate(
            input_ids,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_k=5
        )
    
    # Decodificar e extrair resultado
    full_result = decode(generated[0].tolist())
    
    if "->" in full_result:
        parts = full_result.split("->", 1)
        if len(parts) > 1:
            # Extrair apenas até a primeira quebra de linha
            lbot_command = parts[1].strip().split('\n')[0].strip()
            
            # Limpar: manter apenas caracteres válidos do LBML
            # Válidos: dígitos, D, R, F, B, L, R, ;
            cleaned = ''.join(c for c in lbot_command if c.isdigit() or c in 'DRFBL;')
            
            # Validar formato básico (deve ter pelo menos D ou R seguido de número)
            if cleaned and (cleaned[0] in 'DR'):
                return cleaned
    
    return "ERRO"

# Testes do modelo V5.1 com RASA
test_commands = [
    "vá 40 centímetros para frente",
    "gire 90 graus para direita",
    "ande 25 centímetros para frente, depois vire 90 graus à esquerda",
    "mova-se 50 centímetros para trás",
    "gire 180 graus para esquerda",
    "ande 30 centímetros para direita e depois gire 45 graus para direita",
    "vá 15 centímetros à esquerda, depois ande 20 centímetros para frente",
    "desloque-se 10 centímetros atrás e depois gire 60 graus para esquerda"
]

print("🧪 TESTANDO TRADUTOR LBOT V5.1 com RASA\n")
print("Formato LBML: D=Deslocamento, R=Rotação, F/B/L/R=Direções, ;=Separador")
print("✨ RASA captura relações: frente↔F, trás↔B, esquerda↔L, direita↔R\n")

for i, cmd in enumerate(test_commands, 1):
    result = lbot_translator(cmd)
    print(f"{i:2d}. '{cmd}'")
    print(f"    → '{result}'\n")

print("✅ Testes V5.1 concluídos!")
print("🔬 Compare estes resultados com V5 base para avaliar ganho do RASA")

## 🎮 7. Interface Interativa

In [ ]:
def interactive_translator():
    """Interface interativa para testar o tradutor V5.1"""
    print("🤖 === TRADUTOR LBOT V5.1 com RASA INTERATIVO ===")
    print("✨ Com Relational-Aware Self-Attention")
    print("Digite comandos em português ou 'sair' para terminar")
    print("Exemplos: 'vá 30 centímetros para frente', 'ande 20 centímetros para direita e depois gire 90 graus'\n")
    
    while True:
        try:
            command = input("🗣️  Comando: ").strip()
            
            if command.lower() in ['sair', 'exit', 'quit', '']:
                print("👋 Tchau!")
                break
            
            translation = lbot_translator(command)
            print(f"🤖 LBot: {translation}\n")
            
        except KeyboardInterrupt:
            print("\n👋 Tchau!")
            break
        except Exception as e:
            print(f"❌ Erro: {e}\n")

# Executar interface interativa
# interactive_translator()  # Descomente para usar

print("✅ Interface interativa V5.1 definida!")
print("💡 Descomente a linha acima para usar a interface")
print("🔬 RASA melhora comandos compostos com múltiplas direções espaciais")

## 📊 8. Análise das Relações Aprendidas (RASA)

Esta seção permite visualizar as relações espaciais que a camada RASA aprendeu durante o treinamento.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_relational_bias():
    """
    Analisa e visualiza os bias relacionais aprendidos pela RASA
    Mostra quais tokens têm relações mais fortes entre si
    """
    model.eval()
    
    # Extrair matriz de bias relacional da primeira camada RASA
    first_rasa = model.transformer.h[0].rasa
    relation_matrix = first_rasa.relation_bias.detach().cpu().numpy()
    
    print("🔍 Analisando relações aprendidas pela RASA...\n")
    print(f"📊 Matriz de relações: {relation_matrix.shape} (vocab_size × vocab_size)")
    
    # Encontrar tokens importantes do LBML
    important_chars = ['F', 'B', 'L', 'R', 'D', ';', '0', '1', '2', '3', '4', '5']
    important_indices = []
    important_labels = []
    
    for char in important_chars:
        if char in stoi:
            important_indices.append(stoi[char])
            important_labels.append(char)
    
    # Extrair submatriz para tokens importantes
    if len(important_indices) > 0:
        important_matrix = relation_matrix[important_indices][:, important_indices]
        
        # Visualizar matriz de relações
        plt.figure(figsize=(10, 8))
        sns.heatmap(important_matrix, 
                    xticklabels=important_labels, 
                    yticklabels=important_labels,
                    cmap='RdBu_r', 
                    center=0,
                    annot=True, 
                    fmt='.2f',
                    cbar_kws={'label': 'Força da Relação'})
        plt.title('Relações Espaciais Aprendidas pela RASA\n(Camada 1)', fontsize=14, fontweight='bold')
        plt.xlabel('Token Destino', fontsize=12)
        plt.ylabel('Token Origem', fontsize=12)
        plt.tight_layout()
        plt.show()
        
        # Encontrar relações mais fortes
        print("\n🔝 Top 10 relações mais fortes (positivas):")
        flat_indices = np.argsort(important_matrix.flatten())[-10:][::-1]
        for idx in flat_indices:
            i, j = divmod(idx, len(important_labels))
            if i != j:  # Ignorar diagonal
                print(f"   {important_labels[i]} → {important_labels[j]}: {important_matrix[i, j]:.3f}")
        
        print("\n🔝 Top 10 relações mais fortes (negativas):")
        flat_indices = np.argsort(important_matrix.flatten())[:10]
        for idx in flat_indices:
            i, j = divmod(idx, len(important_labels))
            if i != j:  # Ignorar diagonal
                print(f"   {important_labels[i]} → {important_labels[j]}: {important_matrix[i, j]:.3f}")
    
    print("\n💡 Interpretação:")
    print("   • Valores POSITIVOS: Tokens frequentemente aparecem juntos")
    print("   • Valores NEGATIVOS: Tokens raramente aparecem juntos")
    print("   • Ex: 'D' e 'F' positivo = deslocamento frente comum")
    print("   • Ex: 'F' e 'B' negativo = frente e trás são opostos")

# Executar análise
# analyze_relational_bias()  # Descomente após treinar

print("✅ Função de análise RASA definida!")
print("💡 Descomente 'analyze_relational_bias()' após o treinamento")

## 📈 9. Comparação V5 vs V5.1

Métricas para avaliar se RASA realmente melhora os resultados.

In [ ]:
def compare_models_performance():
    """
    Compara performance do modelo V5.1 (RASA) com baseline
    Use depois de treinar ambos V5 e V5.1 no mesmo dataset
    """
    
    # Casos de teste focados em comandos compostos com direções múltiplas
    test_cases = [
        # Simples - Frente/Trás
        ("vá 40 centímetros para frente", "D40F;"),
        ("ande 30 centímetros para trás", "D30B;"),
        
        # Simples - Esquerda/Direita
        ("mova-se 25 centímetros para esquerda", "D25L;"),
        ("desloque-se 15 centímetros para direita", "D15R;"),
        
        # Simples - Rotação
        ("gire 90 graus para direita", "R90R;"),
        ("vire 45 graus para esquerda", "R45L;"),
        
        # Compostos - 2 ações (CRÍTICO para RASA)
        ("vá 20 centímetros para frente e depois gire 90 graus para direita", "D20F;R90R;"),
        ("ande 30 centímetros para esquerda, depois vire 45 graus à direita", "D30L;R45R;"),
        ("gire 90 graus para esquerda e depois ande 40 centímetros para frente", "R90L;D40F;"),
        
        # Compostos - 3 ações (MUITO CRÍTICO para RASA)
        ("vá 10 centímetros frente, gire 90 graus direita, ande 20 centímetros esquerda", "D10F;R90R;D20L;"),
        ("ande 15 centímetros trás, vire 45 graus esquerda, vá 25 centímetros direita", "D15B;R45L;D25R;"),
    ]
    
    print("📊 COMPARAÇÃO DE ACURÁCIA\n")
    print("=" * 80)
    
    correct = 0
    total = len(test_cases)
    
    results = []
    
    for i, (command, expected) in enumerate(test_cases, 1):
        result = lbot_translator(command)
        is_correct = (result == expected)
        correct += is_correct
        
        status = "✅" if is_correct else "❌"
        results.append((command, expected, result, is_correct))
        
        print(f"\n{i:2d}. {status} {command}")
        print(f"    Esperado: {expected}")
        print(f"    Obtido:   {result}")
    
    accuracy = (correct / total) * 100
    
    print("\n" + "=" * 80)
    print(f"🎯 RESULTADO FINAL: {correct}/{total} corretos ({accuracy:.1f}%)")
    print("=" * 80)
    
    # Análise por categoria
    simple_correct = sum(1 for cmd, exp, res, corr in results[:6] if corr)
    compound_2_correct = sum(1 for cmd, exp, res, corr in results[6:9] if corr)
    compound_3_correct = sum(1 for cmd, exp, res, corr in results[9:] if corr)
    
    print(f"\n📈 ANÁLISE POR CATEGORIA:")
    print(f"   • Comandos Simples (6): {simple_correct}/6 ({simple_correct/6*100:.1f}%)")
    print(f"   • Compostos 2 ações (3): {compound_2_correct}/3 ({compound_2_correct/3*100:.1f}%)")
    print(f"   • Compostos 3 ações (2): {compound_3_correct}/2 ({compound_3_correct/2*100:.1f}%)")
    
    print(f"\n💡 RASA deve melhorar mais em:")
    print(f"   • Comandos compostos (2-3 ações)")
    print(f"   • Comandos com múltiplas mudanças de direção")
    print(f"   • Sequências espaciais complexas")
    
    print(f"\n🔬 Para comparar com V5:")
    print(f"   1. Treine V5 e V5.1 no mesmo dataset")
    print(f"   2. Execute esta função em ambos")
    print(f"   3. Compare acurácia geral e por categoria")
    print(f"   4. V5.1 deve ter +2-5% em compostos se RASA funcionar")
    
    return accuracy, results

# Executar comparação
# accuracy, results = compare_models_performance()

print("✅ Função de comparação definida!")
print("💡 Execute após treinar para avaliar ganho do RASA")